# Nyra IT V6 — cinque voci e fondo continuo
Due modelli: **Nira** e **Ehi/Ei/Hei/Hey Nira**, anche unite **Ehinira/Einira/Heinira/Heynira**. Solo TTS italiano: Piper Paola/Riccardo/Serena e Kokoro Sara/Nicola. Le grafie non garantiscono pronunce foneticamente distinte.

**Accettazione automatica esplicitamente richiesta da Nicola. Nessuna domanda o verifica d’ascolto.** Restano controlli automatici di integrità/durata, copertura varianti/voci, spazio disco e pipeline. Le anteprime sono consultabili ma non bloccano il run.

Correzione V5: fondo sintetico continuo anche prima e dopo la parola, PRIMA dell’estrazione delle feature. Tutte le classi ricevono la stessa politica. Validation/test positivi coprono 7 profili × 3 livelli di fondo (-75/-60/-45 dBFS). Non rappresentano angoli o metri reali.

40.000 positivi training, due training sequenziali. L4 preferibile, T4 ammessa. Controlli spazio e riserva 8 GiB; servono diverse ore, durata non garantita. Nuovo RUN_ID: non riusa pesi V5. Nessun flash automatico. Più voci non garantiscono generalizzazione alla voce umana.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, shutil, hashlib, importlib.util
RUN_ID = 'nyra-it-v6-004-continuous'
SMOKE = False
USE_DRIVE = True
BACKUP_FOLDER = 'wakeword_training_nyra_it_v4'
RESUME = False  # True solo per un training interrotto dello stesso run.
ROOT = Path('/content/nyra_v6') / RUN_ID
ROOT.mkdir(parents=True, exist_ok=True)
VENV = Path('/content/nyra_v4_env')
PYTHON = str(VENV / 'bin/python')
assert RUN_ID and all(c.isalnum() or c in '-_' for c in RUN_ID)
print('Python Colab:', sys.version)
print('Disco libero:', round(shutil.disk_usage('/content').free/2**30,1), 'GiB')


## Ambiente isolato Python 3.12

In [ ]:
# Python del kernel e Python del training sono indipendenti.
# Usa sempre 3.12 per conservare le dipendenze TensorFlow previste.
VENV = Path('/content/nyra_v4_env_py312')
PYTHON = str(VENV / 'bin/python')
if not Path(PYTHON).exists():
    if sys.version_info[:2] == (3,12):
        subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
    else:
        UV_DIR = Path('/content/nyra_uv_bootstrap')
        UV = str(UV_DIR / 'bin/uv')
        if not Path(UV).exists():
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(UV_DIR),
                            'uv==0.9.7'], check=True)
        uv_env = os.environ.copy()
        uv_env['UV_PYTHON_INSTALL_DIR'] = '/content/nyra_python_managed'
        uv_env['UV_CACHE_DIR'] = '/content/nyra_uv_cache'
        subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)],
                       check=True, env=uv_env)
training_version = subprocess.check_output(
    [PYTHON, '-c', 'import sys; print(str(sys.version_info.major)+"."+str(sys.version_info.minor))'],
    text=True).strip()
if training_version != '3.12':
    raise RuntimeError('Ambiente training inatteso: ' + training_version + '. Usa un nuovo runtime.')
print('Python kernel:', sys.version.split()[0], '| Python training:', training_version)
requirements = [
    'tensorflow[and-cuda]==2.19.1', 'keras==3.10.0', 'numpy==2.1.3', 'scipy==1.15.3',
    'piper-tts==1.8.0', 'pymicro-features==2.0.2', 'mmap-ninja==0.9.0',
    'audiomentations==0.43.1', 'datasets==3.6.0', 'ai-edge-litert==1.4.0',
    'huggingface-hub==0.34.4', 'pyyaml==6.0.2', 'webrtcvad-wheels==2.0.14',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
    'git+https://github.com/OHF-Voice/micro-wake-word@4665173cd35f1cff9a61e06fc427f124766c488e',
]
req_hash = hashlib.sha256(json.dumps(requirements).encode()).hexdigest()
stamp = VENV / 'nyra_requirements.sha256'
if not stamp.exists() or stamp.read_text() != req_hash:
    subprocess.run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
    subprocess.run([PYTHON, '-m', 'pip', 'install'] + requirements, check=True)
    subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
    stamp.write_text(req_hash)
freeze = subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True)
(ROOT / 'environment.txt').write_text(freeze)
subprocess.run(['nvidia-smi'], check=False)
print('Ambiente isolato pronto.')

# Expose pip-installed CUDA libraries to every isolated training subprocess.
site_dir=Path(subprocess.check_output([PYTHON,'-c','import site; print(site.getsitepackages()[0])'],text=True).strip())
CUDA_LIBS=[str(p) for p in (site_dir/'nvidia').glob('*/lib')]
os.environ['LD_LIBRARY_PATH']=os.pathsep.join(CUDA_LIBS+[os.environ.get('LD_LIBRARY_PATH','')])
os.environ['PATH']=str(site_dir/'nvidia/cuda_nvcc/bin')+os.pathsep+os.environ['PATH']

# Kokoro CPU: avoids a second CUDA stack and GPU competition with TensorFlow.
subprocess.run([PYTHON,'-m','pip','install','torch==2.8.0','--index-url','https://download.pytorch.org/whl/cpu'],check=True)
subprocess.run([PYTHON,'-m','pip','install','kokoro==0.9.4'],check=True)
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','espeak-ng'],check=True)
subprocess.run([PYTHON,'-m','pip','check'],check=True)
(ROOT/'environment.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))


## Worker completo e configurazione ufficiale fissata

In [ ]:
WORKER_SOURCE = '"""Nyra IT V4 worker; embedded verbatim in the standalone Colab notebook."""\nimport argparse\nimport collections\nimport hashlib\nimport itertools\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport wave\n\nQUANTIZATION_DRIVER = \'"""Align only quantization representative data; delegate training/conversion to upstream."""\\nimport runpy\\nfrom microwakeword import utils\\noriginal_convert=utils.convert_saved_model_to_tflite\\n\\ndef aligned_convert(config,*args,**kwargs):\\n    cfg=dict(config)\\n    stride=int(cfg[\\\'stride\\\']);length=int(cfg[\\\'spectrogram_length\\\'])\\n    cfg[\\\'spectrogram_length\\\']=((length+stride-1)//stride)*stride\\n    if cfg[\\\'spectrogram_length\\\']!=length:\\n        print(\\\'Quantization representative frames:\\\',length,\\\'->\\\',cfg[\\\'spectrogram_length\\\'],flush=True)\\n    return original_convert(cfg,*args,**kwargs)\\n\\nutils.convert_saved_model_to_tflite=aligned_convert\\nrunpy.run_module(\\\'microwakeword.model_train_eval\\\',run_name=\\\'__main__\\\')\\n\'\n\nPHRASES = {\'ehi_nira_it\': \'Ehi Nira\', \'nira_it\': \'Nira\'}\nMWW_COMMIT = \'4665173cd35f1cff9a61e06fc427f124766c488e\'\nVOICE_REV = \'5041a7a49ae5b3a0f30e6837769eb03fdbb231bd\'\nVOICES = {\'paola\': \'it/it_IT/paola/medium/it_IT-paola-medium.onnx\',\n          \'riccardo\': \'it/it_IT/riccardo/x_low/it_IT-riccardo-x_low.onnx\'}\nFAMILIES = {f\'{v}-{name}\': dict(voice=v, length_scale=length, noise_scale=noise, noise_w_scale=.8)\n            for v in VOICES for name, length, noise in [(\'natural\',1.,.667),(\'fast\',.9,.667),\n                                                       (\'slow\',1.1,.667),(\'clear\',1.,.55)]}\nHARD = [\'Nina\',\'Mira\',\'Nera\',\'Gira\',\'Tira\',\'Lira\',\'Mina\',\'Niro\',\'Nura\',\'Vira\',\'Ira\',\'Nino\',\'Mila\',\n        \'Ehi Nina\',\'Ehi Mira\',\'Ehi Nera\',\'Ehi Lina\',\'Ehi Gina\',\'Ehi\',\'Niente\',\'finirà\',\'venire\',\n        \'La mira è precisa\',\'Gira la pagina\',\'Tira la tenda\',\'La sera è tranquilla\',\n        \'Hai visto Nina?\',\'Una lira italiana\',\'Mi gira la testa\',\'Apri la finestra\']\n# Disjoint templates by split; same hard confusables intentionally recur with fresh TTS.\nTEMPLATES = {\n \'training\': [\'Oggi preparo {obj} in cucina.\', \'Per favore porta {obj} sul tavolo.\',\n              \'Domani compriamo {obj} al mercato.\', \'Hai trovato {obj} nel cassetto?\',\n              \'Questa mattina ho cercato {obj}.\',\'Vorrei parlare di {obj} con te.\'],\n \'validation\': [\'Nel pomeriggio sistemeremo {obj}.\',\'La scatola contiene {obj}.\',\n                \'Puoi spostare {obj} vicino alla porta?\'],\n \'testing\': [\'Durante la settimana useremo {obj}.\',\'Ho lasciato {obj} accanto al divano.\',\n             \'Mi hanno raccontato una storia su {obj}.\']}\nOBJECTS = [\'il pane\',\'il libro\',\'il telefono\',\'il bicchiere\',\'il giornale\',\'il cappotto\',\n           \'il quaderno\',\'il pacco\',\'il piatto\',\'il cuscino\',\'il calendario\',\'il biglietto\',\n           \'il regalo\',\'il vaso\',\'il computer\',\'il documento\',\'il tavolino\',\'il disegno\',\n           \'il caricatore\',\'il portafoglio\',\'il cestino\',\'il telecomando\']\nDEFAULT = dict(run_id=\'nyra-it-v4-001\', seed=18473, reserve_gib=8,\n               positives={\'ehi_nira_it\':6000,\'nira_it\':4000},\n               val_positive=300, test_positive=500, train_hard=4000, val_hard=400,test_hard=600,\n               train_speech=5000,val_speech=300,test_speech=500,train_noise=1500,\n               ambient_validation_minutes=30,ambient_testing_minutes=120,\n               training_steps=[25000,20000],batch_size=128,target_faph=.5,min_recall=.85,\n               tensor_arena_estimate=100000,author=\'Nicola\')\n\ndef digest(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False).encode()).hexdigest()\n\ndef file_hash(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef save(p,obj):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    tmp=p.with_suffix(p.suffix+\'.tmp\');tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False));tmp.replace(p)\n\ndef load(p):return json.loads(Path(p).read_text())\n\ndef select_families(rows,ratings):\n    if set(ratings)!=set(r[\'id\'] for r in rows) or any(type(v) is not bool for v in ratings.values()):\n        raise ValueError(\'Valuta tutti i sample con accetta/scarta.\')\n    result={}\n    for phrase in PHRASES:\n        groups=collections.defaultdict(list)\n        for r in rows:\n            if r[\'phrase\']==phrase:groups[r[\'family\']].append(ratings[r[\'id\']])\n        result[phrase]=sorted(f for f,votes in groups.items() if votes and all(votes))\n        if not result[phrase]:raise ValueError(f\'Nessuna famiglia interamente approvata per {PHRASES[phrase]}.\')\n    return result\n\ndef check_approval(payload,approval):\n    if approval.get(\'confirmed\') is not True or approval.get(\'fingerprint\')!=digest(payload):\n        raise ValueError(\'Approvazione assente o obsoleta: rifai preview/selezione/conferma.\')\n\ndef event_times(probs,cutoff,window,dt=.03,refractory=2.,warmup=3.2):\n    """Offline event policy: moving mean, strict > cutoff, warmup, fixed refractory.\n\n    Not a bit-exact ESPHome simulator (firmware cool-off is data dependent).\n    """\n    times=[];buf=collections.deque(maxlen=window);last=-math.inf\n    for i,p in enumerate(probs):\n        buf.append(float(p));t=(i+1)*dt\n        if len(buf)==window and t>=warmup and sum(buf)>cutoff*window+1e-12 and t-last>=refractory-1e-9:\n            times.append(round(t,6));last=t\n    return times\n\ndef merge_events(a,b,refractory=2.):\n    out=[]\n    for t in sorted(a+b):\n        if not out or t-out[-1]>=refractory-1e-9:out.append(t)\n    return out\n\ndef evaluation_variants(audio,kind,seed):\n    """Deterministic synthetic conditions; derivatives stay within their source split."""\n    import numpy as np\n    if kind!=\'positive\':return [(\'original\',audio)]\n    rng=np.random.default_rng(seed)\n    a=audio.astype(np.float32)/32768\n    noise=rng.normal(0,1,len(a)).astype(np.float32)\n    rms=float(np.sqrt(np.mean(a*a)))\n    noise=noise/max(float(np.sqrt(np.mean(noise*noise))),1e-8)*rms/(10**(8/20))\n    echo=a.copy();delay=1600\n    if len(a)>delay:echo[delay:]+=.3*a[:-delay]\n    def pcm(x):return np.clip(x*32767,-32768,32767).astype(np.int16)\n    return [(\'clean\',audio),(\'quiet\',pcm(a*.15)),(\'noise_snr8\',pcm(a+noise)),(\'echo\',pcm(echo*.7))]\n\n\ndef recall(rows,phrase,cutoff,window):\n    positives=[r for r in rows if r[\'kind\']==\'positive\' and r[\'phrase\']==phrase]\n    if not positives:raise ValueError(\'Mancano positivi per la valutazione.\')\n    # 4 s pre-roll; accept only within the annotated utterance interval + 1 s tail.\n    return sum(any(r.get(\'start\',4.)<=t<=r.get(\'end\',100.)+1.\n                   for t in event_times(r[\'probs\'],cutoff,window)) for r in positives)/len(positives)\n\ndef choose_operating_point(rows,phrase,target_faph=.5,min_recall=.85):\n    if any(r[\'split\']!=\'validation\' for r in rows):raise ValueError(\'Calibrazione ammessa solo su validation.\')\n    ambient=[r for r in rows if r[\'kind\']==\'ambient\']\n    hours=sum(r[\'duration\'] for r in ambient)/3600\n    if not hours:raise ValueError(\'Manca esposizione ambient validation.\')\n    grid=[]\n    for window,cutoff_int in itertools.product([3,5,7],[128,153,179,191,204,217,230,235,240,245,250,253]):\n        cutoff=cutoff_int/255\n        tp=recall(rows,phrase,cutoff,window)\n        events=sum(len(event_times(r[\'probs\'],cutoff,window)) for r in ambient)\n        hard=[r for r in rows if r[\'kind\']==\'hard\']\n        hard_fpr=sum(bool(event_times(r[\'probs\'],cutoff,window)) for r in hard)/max(1,len(hard))\n        grid.append(dict(cutoff=cutoff,window=window,recall=tp,faph=events/hours,\n                         hard_fpr=hard_fpr,validation_hours=hours,\n                         feasible=tp>=min_recall and events/hours<=target_faph))\n    good=[r for r in grid if r[\'feasible\']]\n    best=max(good,key=lambda r:(r[\'recall\'],-r[\'hard_fpr\'],-r[\'faph\'],r[\'cutoff\'],-r[\'window\'])) if good else min(grid,key=lambda r:(r[\'faph\'],r[\'hard_fpr\'],-r[\'recall\']))\n    return dict(best,grid=grid)\n\nclass Run:\n    def __init__(self,root):\n        self.root=Path(root);self.cfg=load(self.root/\'config.json\')\n        self.rng=random.Random(self.cfg[\'seed\']);self.voices={}\n        self.cfg_fingerprint=digest(dict(config=self.cfg,families=FAMILIES,phrases=PHRASES,\n                                         mww=MWW_COMMIT,voices=VOICES,voice_rev=VOICE_REV,worker=file_hash(__file__)))\n    def guard(self,extra=0,label=\'operazione\'):\n        free=shutil.disk_usage(self.root).free/2**30\n        if free<self.cfg[\'reserve_gib\']+extra:\n            raise RuntimeError(f\'{label}: liberi {free:.1f} GiB, richiesti {extra+self.cfg["reserve_gib"]:.1f}. Riduci profilo e rifai preview con nuovo run_id.\')\n    def payload(self):\n        data=load(self.root/\'preview.json\')\n        if data[\'config_fingerprint\']!=self.cfg_fingerprint:raise ValueError(\'Configurazione cambiata: rigenera preview.\')\n        for row in data[\'rows\']:\n            if file_hash(self.root/row[\'path\'])!=row[\'sha256\']:raise ValueError(\'Audio preview modificato: rifai preview.\')\n        return data\n    def approved(self):\n        data=self.payload();a=load(self.root/\'approval.json\');check_approval(data,a)\n        chosen=select_families(data[\'rows\'],a[\'ratings\'])\n        if chosen!=a[\'selected\']:raise ValueError(\'Selezione alterata.\')\n        return chosen\n    def voice(self,name):\n        if name not in self.voices:\n            from piper import PiperVoice\n            self.voices[name]=PiperVoice.load(str(self.root/\'voices\'/Path(VOICES[name]).name),use_cuda=False)\n        return self.voices[name]\n    def synth(self,text,family):\n        import numpy as np\n        from scipy.signal import resample_poly\n        from piper import SynthesisConfig\n        f=FAMILIES[family];voice=self.voice(f[\'voice\'])\n        conf=SynthesisConfig(length_scale=f[\'length_scale\'],noise_scale=f[\'noise_scale\'],\n                             noise_w_scale=f[\'noise_w_scale\'],normalize_audio=True)\n        chunks=list(voice.synthesize(text,syn_config=conf))\n        if not chunks:raise RuntimeError(\'TTS non ha prodotto audio.\')\n        a=np.concatenate([c.audio_float_array for c in chunks]);sr=chunks[0].sample_rate\n        a=resample_poly(a,16000//math.gcd(sr,16000),sr//math.gcd(sr,16000)).astype(np.float32)\n        if not np.isfinite(a).all() or np.max(np.abs(a))<.001:raise RuntimeError(\'Sample vuoto o non valido.\')\n        # Remove digital boundary silence only; never concatenate Ehi and Nira.\n        active=np.flatnonzero(np.abs(a)>.005)\n        if not len(active):raise RuntimeError(\'TTS silenzioso.\')\n        a=a[max(0,active[0]-800):min(len(a),active[-1]+801)]\n        return np.clip(a,-1,1)\n    def write_wav(self,path,a):\n        import numpy as np\n        from scipy.io.wavfile import write\n        path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n        write(str(path),16000,np.clip(a*32767,-32768,32767).astype(np.int16))\n    def ensure_voices(self):\n        from huggingface_hub import hf_hub_download\n        self.guard(2,\'download voci\')\n        for name,remote in VOICES.items():\n            for suffix in [\'\', \'.json\']:\n                src=hf_hub_download(repo_id=\'rhasspy/piper-voices\',filename=remote+suffix,\n                                   revision=VOICE_REV,local_dir=str(self.root/\'voice_download\'))\n                dst=self.root/\'voices\'/(Path(remote).name+suffix);dst.parent.mkdir(exist_ok=True)\n                shutil.copy2(src,dst)\n            card=hf_hub_download(repo_id=\'rhasspy/piper-voices\',filename=str(Path(remote).parent/\'MODEL_CARD\'),\n                                 revision=VOICE_REV,local_dir=str(self.root/\'voice_download\'))\n            shutil.copy2(card,self.root/\'voices\'/f\'{name}_MODEL_CARD.txt\')\n    def preview(self):\n        (self.root/\'approval.json\').unlink(missing_ok=True)\n        self.ensure_voices()\n        rows=[]\n        for phrase,text in PHRASES.items():\n            for family in FAMILIES:\n                for j in range(2):\n                    a=self.synth(text,family)\n                    if len(a)/16000>1.8:raise RuntimeError(f\'{text}/{family}: durata >1.8s; non approvare campioni tagliati.\')\n                    id_=f\'{phrase}__{family}__{j}\'\n                    p=Path(\'preview\')/(id_+\'.wav\');self.write_wav(self.root/p,a)\n                    rows.append(dict(id=id_,phrase=phrase,family=family,path=str(p),sha256=file_hash(self.root/p),duration=len(a)/16000))\n                    print(\'Preview\',id_,flush=True)\n        save(self.root/\'preview.json\',dict(config_fingerprint=self.cfg_fingerprint,rows=rows))\n    def noise(self,seconds,seed):\n        import numpy as np\n        from scipy.signal import lfilter\n        g=np.random.default_rng(seed);n=int(seconds*16000)\n        a=g.normal(0,1,n)\n        mode=seed%5\n        if mode==0:a=lfilter([.1],[1,-.9],a)\n        elif mode==1:a=np.diff(a,prepend=0)\n        elif mode==2:\n            t=np.arange(n)/16000;a=.1*a+np.sin(2*np.pi*(50+seed%100)*t)\n        elif mode==3:\n            a*=.03\n            for _ in range(12):\n                i=int(g.integers(0,max(1,n-2000)));a[i:i+2000]+=np.exp(-np.arange(min(2000,n-i))/180)*g.normal()\n        a=a/(np.max(np.abs(a))+1e-8)*g.uniform(.01,.35)\n        return a.astype(np.float32)\n    def generate(self):\n        import numpy as np\n        chosen=self.approved()\n        self.ensure_voices()\n        if (self.root/\'dataset.json\').exists():\n            d=load(self.root/\'dataset.json\')\n            if d[\'approval_fingerprint\']!=digest(load(self.root/\'approval.json\')):raise ValueError(\'Dataset precedente con altra approvazione: usa nuovo run_id.\')\n            print(\'Dataset già completo: riutilizzato.\');return\n        # Transactional rebuild after interruptions, only THIS run\'s working data.\n        for d in [\'audio\',\'features\',\'scores\',\'models\']:\n            shutil.rmtree(self.root/d,ignore_errors=True)\n        npos=sum(self.cfg[\'positives\'].values())+2*(self.cfg[\'val_positive\']+self.cfg[\'test_positive\'])\n        nneg=sum(self.cfg[k] for k in [\'train_hard\',\'val_hard\',\'test_hard\',\'train_speech\',\'val_speech\',\'test_speech\',\'train_noise\'])\n        wav_gib=((npos+nneg)*5*32000+(self.cfg[\'ambient_validation_minutes\']+self.cfg[\'ambient_testing_minutes\'])*60*32000)/2**30\n        self.guard(wav_gib*1.3+3,\'stima WAV completa\')\n        rows=[];seen={r[\'sha256\'] for r in self.payload()[\'rows\']}\n        def add(kind,split,phrase,family,text,a):\n            if len(rows)%100==0:self.guard(1,f\'generazione {len(rows)}\');print(\'WAV\',len(rows),flush=True)\n            p=Path(\'audio\')/kind/phrase/split/f\'{len(rows):07d}.wav\'\n            self.write_wav(self.root/p,a);h=file_hash(self.root/p)\n            if h in seen:raise RuntimeError(\'Duplicato audio esatto (anche fra split): dataset bloccato. Non riusare la preview.\')\n            seen.add(h)\n            rows.append(dict(id=len(rows),kind=kind,split=split,phrase=phrase,family=family,text=text,\n                             path=str(p),sha256=h,duration=len(a)/16000))\n        for phrase,text in PHRASES.items():\n            for split,n in [(\'training\',self.cfg[\'positives\'][phrase]),(\'validation\',self.cfg[\'val_positive\']),(\'testing\',self.cfg[\'test_positive\'])]:\n                for i in range(n):\n                    family=chosen[phrase][i%len(chosen[phrase])];a=self.synth(text,family)\n                    if len(a)/16000>1.8:raise RuntimeError(f\'Sample positivo troppo lungo: {phrase}/{family}. Rivedi famiglie.\')\n                    add(\'positive\',split,phrase,family,text,a)\n        all_families=sorted(set(sum(chosen.values(),[])))\n        for kind in [\'hard\',\'speech\']:\n            for split,key in [(\'training\',\'train\'),(\'validation\',\'val\'),(\'testing\',\'test\')]:\n                for i in range(self.cfg[f\'{key}_{kind}\']):\n                    text=HARD[i%len(HARD)] if kind==\'hard\' else self.rng.choice(TEMPLATES[split]).format(obj=self.rng.choice(OBJECTS))\n                    # Neither approved wake phrase is ever a negative.\n                    assert \'nira\' not in text.casefold()\n                    fam=self.rng.choice(all_families);a=self.synth(text,fam)\n                    add(kind,split,\'negative\',fam,text,a)\n        for i in range(self.cfg[\'train_noise\']):\n            add(\'noise\',\'training\',\'negative\',\'procedural\',\'\',self.noise(3.2,self.cfg[\'seed\']+i))\n        # Long independent negative streams, not repetitions of training WAVs.\n        for split in [\'validation\',\'testing\']:\n            total=60*self.cfg[f\'ambient_{split}_minutes\'];duration=60\n            for stream in range(math.ceil(total/duration)):\n                seconds=min(duration,total-stream*duration)\n                seed=self.cfg[\'seed\']+(100000 if split==\'validation\' else 200000)+stream\n                rng=random.Random(seed);a=self.noise(seconds,seed)*.08;cursor=.5\n                while cursor<seconds-6:\n                    text=rng.choice(TEMPLATES[split]).format(obj=rng.choice(OBJECTS))\n                    if rng.random()<.2:text=rng.choice(HARD)\n                    fam=rng.choice(all_families);utterance=self.synth(text,fam)*rng.uniform(.15,.7)\n                    i=int(cursor*16000);length=min(len(utterance),len(a)-i)\n                    a[i:i+length]+=utterance[:length];cursor+=length/16000+rng.uniform(.3,1.8)\n                add(\'ambient\',split,\'negative\',\'mixed\',\'TTS italiano + rumori procedurali\',np.clip(a,-1,1))\n        save(self.root/\'dataset.json\',dict(approval_fingerprint=digest(load(self.root/\'approval.json\')),rows=rows))\n        print(\'Dataset completo\',len(rows),\'WAV; nessun corpus registrato.\',flush=True)\n    def dataset(self):\n        self.approved();d=load(self.root/\'dataset.json\')\n        if d[\'approval_fingerprint\']!=digest(load(self.root/\'approval.json\')):raise ValueError(\'Dataset obsoleto.\')\n        return d[\'rows\']\n    def features(self):\n        import numpy as np\n        from scipy.io.wavfile import read\n        from microwakeword.audio.audio_utils import generate_features_for_clip\n        from microwakeword.audio.augmentation import Augmentation\n        from mmap_ninja.ragged import RaggedMmap\n        rows=self.dataset()\n        # Direct WAV iteration avoids datasets.Audio API/codec drift; official augmentation,\n        # official microfrontend, official RaggedMmap, no resplitting inside Clips.\n        aug=Augmentation(augmentation_duration_s=None,augmentation_probabilities={\n            \'SevenBandParametricEQ\':.2,\'TanhDistortion\':.1,\'PitchShift\':.15,\n            \'BandStopFilter\':.1,\'AddColorNoise\':.45,\'Gain\':1.,\'GainTransition\':.25},\n            impulse_paths=[],background_paths=[],min_jitter_s=.10,max_jitter_s=.25,\n            min_gain_db=-28,max_gain_db=-1)\n        for kind,phrase,split in sorted(set((r[\'kind\'],r[\'phrase\'],r[\'split\']) for r in rows)):\n            subset=[r for r in rows if (r[\'kind\'],r[\'phrase\'],r[\'split\'])==(kind,phrase,split)]\n            source=phrase if kind==\'positive\' else kind\n            outsplit=(split+\'_ambient\') if kind==\'ambient\' else split\n            out=self.root/\'features\'/source/outsplit/\'wakeword_mmap\'\n            marker=out.parent/\'complete.json\'\n            fingerprint=digest(dict(rows=subset,config=self.cfg_fingerprint))\n            if marker.exists() and load(marker).get(\'fingerprint\')==fingerprint and out.exists():\n                print(\'Feature già complete\',source,outsplit);continue\n            # Conservative exact shape-based upper bound including mmap overhead and safety.\n            seconds=sum((3.2 if kind==\'positive\' else max(3.2,r[\'duration\'])) for r in subset)\n            repeats=2 if split==\'training\' else 1\n            estimate=seconds*100*40*4*repeats/2**30*1.5\n            self.guard(estimate+1,f\'feature {source}/{outsplit}\')\n            shutil.rmtree(out,ignore_errors=True);marker.unlink(missing_ok=True);out.parent.mkdir(parents=True,exist_ok=True)\n            def iterator():\n                for r in subset:\n                    sr,a=read(self.root/r[\'path\']);assert sr==16000 and a.dtype==np.int16\n                    for _ in range(repeats):\n                        if kind==\'ambient\':b=a\n                        elif split==\'training\':\n                            # Same acoustic augmentation for positives and negatives; no class cue.\n                            b=aug.augment_clip(a.astype(np.float32)/32768)\n                            b=continuous_context(b,self.cfg[\'seed\']+r[\'id\']*7919+_,tail_s=.15)\n                        else:\n                            # Explicit deterministic right alignment, untouched pronunciation.\n                            b=continuous_context(a,self.cfg[\'seed\']+r[\'id\']*7919,tail_s=.15)\n                        x=generate_features_for_clip(b,step_ms=10)\n                        if x.ndim!=2 or x.shape[1]!=40 or not np.isfinite(x).all():raise RuntimeError(\'Feature non valide.\')\n                        yield x\n            RaggedMmap.from_generator(out_dir=str(out),sample_generator=iterator(),batch_size=64,verbose=True)\n            save(marker,dict(fingerprint=fingerprint,count=len(subset)*repeats))\n            self.guard(0,\'fine estrazione\')\n        print(\'Features condivise: negativi una sola copia per i due modelli.\')\n    def train_config(self,phrase):\n        def src(name,truth,weight,penalty=1.,strategy=\'random\'):\n            return dict(features_dir=str(self.root/\'features\'/name),truth=truth,sampling_weight=weight,\n                        penalty_weight=penalty,truncation_strategy=strategy,type=\'mmap\')\n        return dict(window_step_ms=10,train_dir=str(self.root/\'models\'/phrase),\n                    features=[src(phrase,True,8.,2.,\'truncate_start\'),src(\'hard\',False,10.,4.),\n                              src(\'speech\',False,12.,2.),src(\'noise\',False,5.),src(\'continuous\',False,12.,2.),src(\'ambient\',False,0.,1.,\'split\')],\n                    training_steps=self.cfg[\'training_steps\'],positive_class_weight=[2,2],\n                    negative_class_weight=[20,30],learning_rates=[.001,.0001],batch_size=self.cfg[\'batch_size\'],\n                    time_mask_max_size=[5,5],time_mask_count=[1,1],freq_mask_max_size=[3,3],freq_mask_count=[1,1],\n                    eval_step_interval=500,clip_duration_ms=1800,target_minimization=self.cfg[\'target_faph\'],\n                    minimization_metric=\'ambient_false_positives_per_hour\',maximization_metric=\'average_viable_recall\')\n    def train(self,phrase,resume=False):\n        import yaml\n        self.dataset();self.guard(4,\'training \'+phrase)\n        cfg=self.train_config(phrase);p=self.root/f\'{phrase}_training.yaml\'\n        p.write_text(yaml.safe_dump(cfg,sort_keys=False))\n        model_dir=self.root/\'models\'/phrase;done=model_dir/\'complete.json\'\n        if done.exists() and load(done).get(\'config\')==digest(cfg):\n            print(\'Training già completato:\',phrase);return\n        if model_dir.exists() and any(model_dir.iterdir()) and not resume:\n            raise RuntimeError(\'Training interrotto presente. Per riprendere riesegui la cella con RESUME=True; il calendario training ufficiale riparte. Non cancelliamo checkpoint.\')\n        # The official trainer rejects a precreated log directory unless restore is enabled.\n        # Fresh-run/interruption protection above remains in effect; upstream restores\n        # a checkpoint only when one exists.\n        model_dir.mkdir(parents=True,exist_ok=True)\n        cmd=[sys.executable,\'-u\',\'-c\',QUANTIZATION_DRIVER,\'--training_config\',str(p),\n             \'--train\',\'1\',\'--restore_checkpoint\',\'1\',\'--test_tflite_streaming_quantized\',\'1\',\'--use_weights\',\'best_weights\',\n             \'mixednet\',\'--pointwise_filters\',\'64,64,64,64\',\'--repeat_in_block\',\'1,1,1,1\',\n             \'--mixconv_kernel_sizes\',\'[5], [7,11], [9,15], [23]\',\'--residual_connection\',\'0,0,0,0\',\n             \'--first_conv_filters\',\'32\',\'--first_conv_kernel_size\',\'5\',\'--stride\',\'3\']\n        env=os.environ.copy();env[\'XLA_FLAGS\']=\'--xla_gpu_autotune_level=0\';env[\'PYTHONUNBUFFERED\']=\'1\'\n        log=model_dir/\'training.log\';last=time.monotonic()\n        with log.open(\'w\') as f:\n            proc=subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)\n            for line in proc.stdout:\n                print(line,end=\'\',flush=True);f.write(line);f.flush()\n                if time.monotonic()-last>300:self.backup(light=True);last=time.monotonic()\n            code=proc.wait()\n        self.backup(light=True)\n        if code:raise RuntimeError(f\'Training {phrase} fallito, codice {code}. Leggi {log}. Non è stato alterato il codice ufficiale.\')\n        self.model_path(phrase)\n        save(done,dict(config=digest(cfg)));self.backup(light=True)\n    def model_path(self,phrase):\n        p=self.root/\'models\'/phrase/\'tflite_stream_state_internal_quant\'/\'stream_state_internal_quant.tflite\'\n        if not p.is_file():raise FileNotFoundError(p)\n        return p\n    def score(self,split):\n        import numpy as np\n        from scipy.io.wavfile import read\n        from microwakeword.audio.audio_utils import generate_features_for_clip\n        from microwakeword.inference import Model\n        rows=[r for r in self.dataset() if r[\'split\']==split]\n        # Noise training rows are never included in evaluation.\n        for phrase in PHRASES:\n            model_path=self.model_path(phrase);out=self.root/\'scores\'/f\'{phrase}_{split}.json\'\n            fp=digest(dict(model=file_hash(model_path),rows=rows,policy=\'continuous-v6-21conditions\'))\n            if out.exists() and load(out)[\'fingerprint\']==fp:print(\'Score già presenti\',out);continue\n            results=[];start=time.monotonic()\n            for i,r in enumerate(rows):\n                sr,a=read(self.root/r[\'path\']);assert sr==16000\n                for condition,variant in evaluation_variants(a,r[\'kind\'],self.cfg[\'seed\']+r[\'id\']):\n                    # Fresh interpreter and fresh frontend for each INDEPENDENT stream/clip.\n                    model=Model(str(model_path),stride=3)\n                    assert model.input_details[0][\'dtype\']==np.int8 and model.output_details[0][\'dtype\']==np.uint8\n                    assert list(model.input_details[0][\'shape\'])==[1,3,40]\n                    assert list(model.output_details[0][\'shape\'])==[1,1]\n                    padded=variant # 4 s warm-up, 1 s detection tail.\n                    x=generate_features_for_clip(padded,step_ms=10)\n                    probs=list(map(float,model.predict_spectrogram(x)))\n                    results.append(dict(r,**({\'duration\':len(padded)/16000} if r[\'kind\']!=\'positive\' else {}),id=f"{r[\'id\']}:{condition}",condition=condition,probs=probs,start=4.,end=4.+r[\'duration\']))\n                    del model\n                if i%50==0:print(phrase,split,i,\'/\',len(rows),\'elapsed\',round(time.monotonic()-start),\'s\',flush=True)\n            save(out,dict(fingerprint=fp,model_hash=file_hash(model_path),rows=results))\n        self.backup(light=True)\n    def calibrate(self):\n        ops={}\n        for phrase in PHRASES:\n            data=load(self.root/\'scores\'/f\'{phrase}_validation.json\')\n            if data[\'model_hash\']!=file_hash(self.model_path(phrase)):raise ValueError(\'Score validation obsoleti.\')\n            ops[phrase]=choose_operating_point(data[\'rows\'],phrase,self.cfg[\'target_faph\'],self.cfg[\'min_recall\'])\n            ops[phrase][\'model_hash\']=data[\'model_hash\']\n            ops[phrase][\'validation_fingerprint\']=data[\'fingerprint\']\n        save(self.root/\'operating_points.json\',ops)\n        print(json.dumps({k:{kk:vv for kk,vv in v.items() if kk!=\'grid\'} for k,v in ops.items()},indent=2))\n        self.backup(light=True)\n    def report(self):\n        from scipy.stats import chi2\n        self.dataset();ops=load(self.root/\'operating_points.json\');allrows={};report={}\n        for phrase in PHRASES:\n            data=load(self.root/\'scores\'/f\'{phrase}_testing.json\')\n            if data[\'model_hash\']!=file_hash(self.model_path(phrase)):raise ValueError(\'Score test obsoleti.\')\n            rows=data[\'rows\'];allrows[phrase]=rows;o=ops[phrase]\n            if o[\'model_hash\']!=data[\'model_hash\']:raise ValueError(\'Calibrazione obsoleta.\')\n            cut=o[\'cutoff\'];win=o[\'window\']\n            amb=[r for r in rows if r[\'kind\']==\'ambient\'];h=sum(r[\'duration\'] for r in amb)/3600\n            counts=sum(len(event_times(r[\'probs\'],cut,win)) for r in amb)\n            hard=[r for r in rows if r[\'kind\']==\'hard\'];speech=[r for r in rows if r[\'kind\']==\'speech\']\n            report[phrase]=dict(operating_point={k:v for k,v in o.items() if k!=\'grid\'},\n                recall_by_phrase={p:recall(rows,p,cut,win) for p in PHRASES},\n                recall_by_condition={c:{p:recall([r for r in rows if r.get(\'condition\')==c],p,cut,win) for p in PHRASES}\n                                     for c in EVAL_CONDITIONS},\n                hard_false_clip_fraction=sum(bool(event_times(r[\'probs\'],cut,win)) for r in hard)/len(hard),\n                speech_false_clip_fraction=sum(bool(event_times(r[\'probs\'],cut,win)) for r in speech)/len(speech),\n                ambient_events=counts,ambient_hours=h,synthetic_faph=counts/h,\n                poisson_upper95_faph=float(.5*chi2.ppf(.95,2*(counts+1))/h),\n                model_bytes=self.model_path(phrase).stat().st_size)\n        first,second=list(PHRASES);a={r[\'id\']:r for r in allrows[first]};b={r[\'id\']:r for r in allrows[second]}\n        assert a.keys()==b.keys()\n        def events(r,p):return event_times(r[\'probs\'],ops[p][\'cutoff\'],ops[p][\'window\'])\n        n=0;hours=0.;union_tp=collections.Counter();totals=collections.Counter();union_hard=0;nhard=0\n        for id_,r in a.items():\n            ev=merge_events(events(r,first),events(b[id_],second))\n            if r[\'kind\']==\'ambient\':n+=len(ev);hours+=r[\'duration\']/3600\n            if r[\'kind\']==\'positive\':\n                totals[r[\'phrase\']]+=1\n                union_tp[r[\'phrase\']]+=any(r[\'start\']<=t<=r[\'end\']+1. for t in ev)\n            if r[\'kind\']==\'hard\':union_hard+=bool(ev);nhard+=1\n        report[\'union\']=dict(synthetic_faph=n/hours,events=n,hours=hours,\n                            poisson_upper95_faph=float(.5*chi2.ppf(.95,2*(n+1))/hours),\n                            recall_by_phrase={p:union_tp[p]/totals[p] for p in PHRASES},hard_false_clip_fraction=union_hard/nhard)\n        report[\'operating_points_hash\']=digest(ops)\n        report[\'model_hashes\']={p:file_hash(self.model_path(p)) for p in PHRASES}\n        report[\'limitations\']=[\'Solo due voci TTS; nessun test con parlanti umani o audio domestico.\',\n            \'Gli split separano waveform, seed e template di frasi; non separano il modello TTS né tutti i fonemi.\',\n            \'FP/h sintetici: non rappresentano la distribuzione domestica; limite Poisson solo descrittivo e sotto ipotesi di eventi indipendenti.\',\n            \'Eventi offline: media mobile e refractory 2 s, warm-up 3.2 s; non replica esattamente cool-off ESPHome, VAD, stop/restart e scheduling.\',\n            \'Nira dentro Ehi Nira può attivare il modello corto: risposta positiva prevista, non falso positivo.\',\n            \'Il riconoscitore lungo può imparare il suffisso Nira: recall incrociata riportata, nessuna promessa di isolamento lessicale.\',\n            \'Arena TFLM, RAM complessiva e latenza sul Waveshare ESP32-S3 richiedono compilazione e test sul dispositivo.\']\n        report[\'test_passed_descriptively\']=all(report[p][\'recall_by_phrase\'][p]>=self.cfg[\'min_recall\'] for p in PHRASES) and report[\'union\'][\'synthetic_faph\']<=self.cfg[\'target_faph\']\n        save(self.root/\'evaluation.json\',report);print(json.dumps(report,indent=2,ensure_ascii=False));self.backup(light=True)\n    def export(self):\n        self.dataset();ops=load(self.root/\'operating_points.json\');report=load(self.root/\'evaluation.json\')\n        if report[\'operating_points_hash\']!=digest(ops) or report[\'model_hashes\']!={p:file_hash(self.model_path(p)) for p in PHRASES}:\n            raise ValueError(\'Report non corrispondente a modelli/soglie attuali: rivaluta.\')\n        if not all(o[\'feasible\'] for o in ops.values()):\n            raise RuntimeError(\'Nessuna coppia soglie/recall accettabile su validation. Report salvato; rivedi dati o training, senza calibrare sul test.\')\n        out=self.root/\'export\';out.mkdir(exist_ok=True)\n        resources={}\n        for phrase,text in PHRASES.items():\n            src=self.model_path(phrase);shutil.copy2(src,out/f\'{phrase}.tflite\');op=ops[phrase]\n            manifest=dict(type=\'micro\',wake_word=text,author=self.cfg[\'author\'],model=f\'{phrase}.tflite\',\n                          trained_languages=[\'it\'],version=2,micro=dict(probability_cutoff=op[\'cutoff\'],\n                          sliding_window_size=op[\'window\'],feature_step_size=10,\n                          tensor_arena_size=self.cfg[\'tensor_arena_estimate\'],minimum_esphome_version=\'2024.7.0\'))\n            save(out/f\'{phrase}.json\',manifest)\n            resources[phrase]=dict(tflite_bytes=src.stat().st_size,arena_estimate=self.cfg[\'tensor_arena_estimate\'],\n                                    variable_arena_reference=1024,arena_measured=False)\n        resources[\'notes\']=\'Due arene + overhead/framework/audio; dimensione .tflite non misura RAM. Budget arena candidato 2 x 100000 B + variabili, da misurare. Test prima lungo da solo, poi corto da solo, poi entrambi.\'\n        save(out/\'resources.json\',resources)\n        for name in [\'config.json\',\'approval.json\',\'preview.json\',\'evaluation.json\',\'operating_points.json\',\'environment.txt\',\'source_provenance.json\']:\n            p=self.root/name\n            if p.exists():shutil.copy2(p,out/name)\n        for p in self.root.glob(\'*_training.yaml\'):shutil.copy2(p,out/p.name)\n        for p in (self.root/\'voices\').glob(\'*MODEL_CARD*\'):shutil.copy2(p,out/p.name)\n        yaml_text=\'\'\'# Merge into EXISTING Nyra package; replaces the two model overrides.\n# Keep existing microphone, mww ID and on_wake_word_detected automation.\nmicro_wake_word:\n  models:\n    - id: !extend alexa\n      model: /config/esphome/models/ehi_nira_it.json\n    - id: !extend okay_nabu\n      model: /config/esphome/models/nira_it.json\n# Enable BOTH models explicitly in an existing on_boot (after priority -100):\n# - micro_wake_word.enable_model: alexa\n# - micro_wake_word.enable_model: okay_nabu\n# Existing model enabled states can persist in flash; verify both in HA/logs.\n\'\'\'\n        (out/\'nyra-existing-package.yaml\').write_text(yaml_text)\n        (out/\'README.txt\').write_text(\'MODELLI CANDIDATI, SOLO TEST SINTETICI. Leggere evaluation.json e resources.json.\\n\'\n            \'Manifest schema 2. Copiare ogni coppia JSON/TFLite nello stesso percorso.\\n\'\n            \'Arena stimata, non misurata; compilare ESPHome e verificare allocazioni e latenza sul dispositivo.\\n\'\n            \'Non sostituire alla cieca il file completo dello speaker. YAML fornito per i due !extend esistenti.\\n\'\n            \'Il test finale non viene usato per scegliere soglie: eventuali fallimenti richiedono nuovi dati/run.\\n\')\n        shutil.make_archive(str(self.root/\'nyra_it_v6_candidates\'),\'zip\',out)\n        self.backup(light=True);print(\'ZIP candidati:\',self.root/\'nyra_it_v5_candidates.zip\')\n    def backup(self,light=True):\n        target=os.environ.get(\'NYRA_BACKUP\')\n        if not target:return\n        dst=Path(target)/self.cfg[\'run_id\'];dst.mkdir(parents=True,exist_ok=True)\n        # No audio/features in Drive; reproducibility records + weights/checkpoints/reports.\n        for p in self.root.glob(\'*\'):\n            if p.is_file():shutil.copy2(p,dst/p.name)\n        for name in [\'preview\',\'models\',\'export\']:\n            src=self.root/name\n            if src.exists():shutil.copytree(src,dst/name,dirs_exist_ok=True)\n        print(\'Backup Drive:\',dst,flush=True)\n\n# V5 dataset and evaluation extension. Official frontend/trainer are unchanged.\nVOICES = dict(VOICES, serena=\'it/it_IT/serena/medium/it_IT-serena-medium.onnx\')\nTEXT_VARIANTS = {\n \'ehi_nira_it\': {\'spaced\':\'Ehi Nira\',\'ei_spaced\':\'Ei Nira\',\'joined_ehi\':\'Ehinira\',\'joined_ei\':\'Einira\',\'pause\':\'Ehi, Nira\'},\n \'nira_it\': {\'solo\':\'Nira\',\'calling\':\'Nira!\',\'question\':\'Nira?\'}}\nFAMILIES = {\n f\'{voice}-{style}-{phrase}-{variant}\':dict(voice=voice,length_scale=length,noise_scale=.667,noise_w_scale=.8,\n     phrase=phrase,variant=variant,text=text,style=style)\n for voice in VOICES for style,length in [(\'natural\',1.),(\'fast\',.82),(\'slow\',1.16)]\n for phrase,variants in TEXT_VARIANTS.items() for variant,text in variants.items()}\nACOUSTIC_PROFILES = (\'clean\',\'quiet\',\'off_axis_mild\',\'off_axis_strong\',\'reflections\',\'noise_snr10\',\'off_axis_noise\')\nHARD = HARD + [\'Ei Nina\',\'Einina\',\'Ehimira\',\'Ehinina\',\'Ehi gira\',\'Ei Mira\',\'Mira!\',\'Nina?\']\nDEFAULT = dict(DEFAULT,run_id=\'nyra-it-v5-003-coverage\',seed=310301,\n positives={\'ehi_nira_it\':18000,\'nira_it\':12000},val_positive=600,test_positive=900,\n train_hard=8000,train_speech=8000,train_noise=2000,train_continuous=1800,\n ambient_validation_minutes=120,ambient_testing_minutes=240,training_steps=[35000,25000],\n acoustic_profiles=list(ACOUSTIC_PROFILES))\n\n\ndef acoustic(audio,profile,seed):\n    """Approximate digital attenuation/off-axis/reflection effects, not calibrated room IRs."""\n    import numpy as np\n    from scipy.signal import butter,sosfilt,fftconvolve\n    a=np.asarray(audio,dtype=np.float32).copy();rng=np.random.default_rng(seed)\n    if profile not in ACOUSTIC_PROFILES:raise ValueError(\'Profilo acustico sconosciuto: \'+profile)\n    if profile==\'clean\':return np.clip(a,-1,1)\n    if profile in (\'off_axis_mild\',\'off_axis_strong\',\'off_axis_noise\'):\n        fc=rng.uniform(2500,4200) if profile==\'off_axis_mild\' else rng.uniform(1000,2200)\n        a=sosfilt(butter(2,fc,fs=16000,output=\'sos\'),a).astype(np.float32)\n        a*=10**(rng.uniform(-6,-2)/20)\n    if profile in (\'quiet\',\'off_axis_noise\'):a*=10**(rng.uniform(-22,-8)/20)\n    if profile==\'reflections\':\n        impulse=np.zeros(7200,dtype=np.float32);impulse[0]=1\n        for _ in range(10):\n            d=int(rng.uniform(.018,.4)*16000)\n            impulse[d]+=rng.uniform(-.35,.35)*np.exp(-d/(16000*.17))\n        a=fftconvolve(a,impulse)[:len(a)].astype(np.float32)\n    if profile in (\'noise_snr10\',\'off_axis_noise\'):\n        snr=10 if profile==\'noise_snr10\' else rng.uniform(5,15)\n        n=rng.normal(0,1,len(a)).astype(np.float32)\n        rms=float(np.sqrt(np.mean(a*a)))\n        n*=rms/(max(float(np.sqrt(np.mean(n*n))),1e-8)*10**(snr/20))\n        a+=n\n    peak=float(np.max(np.abs(a))) if len(a) else 0.\n    if peak>1:a/=peak*1.001\n    if not np.isfinite(a).all():raise ValueError(\'Trasformazione non finita\')\n    return a.astype(np.float32)\n\n\ndef select_families(rows,ratings):\n    if set(ratings)!=set(r[\'id\'] for r in rows) or any(type(v) is not bool for v in ratings.values()):\n        raise ValueError(\'Valuta tutti i sample: nessuna accettazione automatica.\')\n    selected={}\n    for phrase in PHRASES:\n        groups=collections.defaultdict(list)\n        for r in rows:\n            if r[\'phrase\']==phrase and r.get(\'preview_kind\',\'tts\')==\'tts\':groups[r[\'family\']].append(ratings[r[\'id\']])\n        selected[phrase]=sorted(k for k,vs in groups.items() if vs and all(vs))\n        if not selected[phrase]:raise ValueError(\'Nessuna famiglia approvata per \'+phrase)\n    return selected\n\n\n_original_choose_operating_point=choose_operating_point\n\ndef choose_operating_point(rows,phrase,target_faph=.5,min_recall=.85):\n    op=_original_choose_operating_point(rows,phrase,target_faph,min_recall)\n    variants=sorted({r.get(\'variant\',\'canonical\') for r in rows if r[\'kind\']==\'positive\' and r[\'phrase\']==phrase})\n    for item in op[\'grid\']:\n        item[\'recall_by_variant\']={v:recall([r for r in rows if r.get(\'variant\',\'canonical\')==v],phrase,item[\'cutoff\'],item[\'window\']) for v in variants}\n        item[\'feasible\']=item[\'feasible\'] and all(x>=.75 for x in item[\'recall_by_variant\'].values())\n    good=[r for r in op[\'grid\'] if r[\'feasible\']]\n    key=lambda r:(r[\'recall\'],-r[\'hard_fpr\'],-r[\'faph\'],r[\'cutoff\'],-r[\'window\'])\n    best=max(good,key=key) if good else min(op[\'grid\'],key=lambda r:(r[\'faph\'],-min(r[\'recall_by_variant\'].values()),-r[\'recall\']))\n    return dict(best,grid=op[\'grid\'])\n\n\n_original_evaluation_variants=evaluation_variants\n\ndef evaluation_variants(audio,kind,seed):\n    if kind!=\'positive\':return [(\'original\',audio)]\n    import numpy as np\n    # Conditions are applied to the same independent TTS source, never new speakers.\n    a=audio.astype(np.float32)/32768\n    return [(p,np.clip(acoustic(a,p,seed+i*1301)*32767,-32768,32767).astype(np.int16)) for i,p in enumerate(ACTIVE_EVALUATION_PROFILES)]\n\n\nACTIVE_EVALUATION_PROFILES=ACOUSTIC_PROFILES\n\n_BaseRun=Run\n\nclass Run(_BaseRun):\n    def preview(self):\n        (self.root/\'approval.json\').unlink(missing_ok=True);self.ensure_voices();rows=[];unavailable=[]\n        for family,f in FAMILIES.items():\n            a=self.synth(f[\'text\'],family)\n            if len(a)>28800:\n                unavailable.append(dict(family=family,duration=len(a)/16000));continue\n            id_=family;p=Path(\'preview\')/(id_+\'.wav\');self.write_wav(self.root/p,a)\n            rows.append(dict(id=id_,phrase=f[\'phrase\'],family=family,variant=f[\'variant\'],text=f[\'text\'],preview_kind=\'tts\',\n                path=str(p),sha256=file_hash(self.root/p),duration=len(a)/16000))\n        anchor=next(r for r in rows if r[\'phrase\']==\'ehi_nira_it\' and r[\'variant\']==\'spaced\' and FAMILIES[r[\'family\']][\'style\']==\'natural\')\n        from scipy.io.wavfile import read\n        sr,pcm=read(self.root/anchor[\'path\']);assert sr==16000\n        for i,profile in enumerate(ACOUSTIC_PROFILES):\n            a=acoustic(pcm.astype(\'float32\')/32768,profile,self.cfg[\'seed\']+i)\n            id_=\'acoustic-\'+profile;p=Path(\'preview\')/(id_+\'.wav\');self.write_wav(self.root/p,a)\n            rows.append(dict(id=id_,phrase=\'ehi_nira_it\',family=anchor[\'family\'],preview_kind=\'acoustic\',profile=profile,text=\'Ehi Nira\',\n                path=str(p),sha256=file_hash(self.root/p),duration=len(a)/16000))\n        save(self.root/\'unavailable_preview_families.json\',unavailable)\n        save(self.root/\'preview.json\',dict(config_fingerprint=self.cfg_fingerprint,rows=rows))\n        print(\'Anteprime nuove:\',len(rows),\'famiglie non troncate:\',len(unavailable),\'(scartate per durata >1.8s)\',flush=True)\n\n    def approved(self):\n        chosen=super().approved()\n        variants={FAMILIES[f][\'variant\'] for f in chosen[\'ehi_nira_it\']}\n        if not variants.intersection({\'joined_ehi\',\'joined_ei\'}):raise ValueError(\'Serve almeno una pronuncia legata approvata: Ehinira o Einira.\')\n        if not variants.intersection({\'spaced\',\'ei_spaced\'}):raise ValueError(\'Serve almeno una pronuncia naturale separata approvata.\')\n        self.profiles=self.approved_profiles()\n        return chosen\n\n    def approved_profiles(self):\n        payload=self.payload();approval=load(self.root/\'approval.json\');check_approval(payload,approval)\n        result=[r[\'profile\'] for r in payload[\'rows\'] if r.get(\'preview_kind\')==\'acoustic\' and approval[\'ratings\'].get(r[\'id\']) is True]\n        if \'clean\' not in result:raise ValueError(\'Il riferimento acustico pulito deve essere approvato.\')\n        return result\n\n    def generate(self):\n        import numpy as np\n        chosen=self.approved();profiles=self.profiles\n        approval_hash=digest(load(self.root/\'approval.json\'))\n        old=self.root/\'dataset.json\'\n        if old.exists():\n            if load(old).get(\'approval_fingerprint\')!=approval_hash:\n                raise ValueError(\'Approvazione cambiata dopo il dataset: usa un nuovo RUN_ID per non riusare pesi precedenti.\')\n            print(\'Dataset V5 già completo\');return\n        # Gate checked before any deletion or mass synthesis.\n        seconds=sum(self.cfg[\'positives\'].values())*1.8+sum(self.cfg[k] for k in (\'train_hard\',\'train_speech\'))*3+self.cfg[\'train_continuous\']*8\n        self.guard(seconds*32000/2**30*1.5+5,\'stima V5 WAV + dipendenze\')\n        shutil.rmtree(self.root/\'audio\',ignore_errors=True);rows=[];hashes={r[\'sha256\']:\'preview\' for r in self.payload()[\'rows\']}\n        def add(kind,split,phrase,family,text,a,variant=None):\n            id_=len(rows)\n            if id_%100==0:self.guard(.25,\'generazione V5\');print(\'WAV V5\',id_,flush=True)\n            profile=profiles[id_%len(profiles)] if split==\'training\' else \'clean\'\n            a=acoustic(a,profile,self.cfg[\'seed\']+id_*7919)\n            p=Path(\'audio\')/kind/phrase/split/f\'{id_:07d}.wav\';self.write_wav(self.root/p,a)\n            h=file_hash(self.root/p)\n            if h in hashes:raise RuntimeError(\'Waveform identica alla preview o già generata; run fermato.\')\n            hashes[h]=split\n            rows.append(dict(id=id_,kind=kind,split=split,phrase=phrase,family=family,text=text,variant=variant or \'negative\',\n                acoustic_profile=profile,path=str(p),sha256=h,duration=len(a)/16000))\n        for phrase in PHRASES:\n            families=chosen[phrase]\n            for split,n in [(\'training\',self.cfg[\'positives\'][phrase]),(\'validation\',self.cfg[\'val_positive\']),(\'testing\',self.cfg[\'test_positive\'])]:\n                # New stochastic WAVs; round-robin preserves approved text/style coverage in every split.\n                for i in range(n):\n                    family=families[i%len(families)];f=FAMILIES[family]\n                    for duration_attempt in range(16):\n                        a=self.synth(f[\'text\'],family)\n                        if len(a)<=28800:break\n                    else:raise RuntimeError(\'16 sintesi oltre 1.8s: \'+family)\n                    if len(a)>28800:raise RuntimeError(\'Sintesi positiva >1.8s: non troncare; rivedere famiglia \'+family)\n                    add(\'positive\',split,phrase,family,f[\'text\'],a,f[\'variant\'])\n        all_families=sorted(set(sum(chosen.values(),[])))\n        for kind in (\'hard\',\'speech\'):\n            for split,key in [(\'training\',\'train\'),(\'validation\',\'val\'),(\'testing\',\'test\')]:\n                for i in range(self.cfg[f\'{key}_{kind}\']):\n                    text=HARD[i%len(HARD)] if kind==\'hard\' else self.rng.choice(TEMPLATES[split]).format(obj=self.rng.choice(OBJECTS))\n                    assert \'nira\' not in text.casefold()\n                    family=self.rng.choice(all_families);add(kind,split,\'negative\',family,text,self.synth(text,family))\n        for i in range(self.cfg[\'train_continuous\']):\n            a=self.noise(8.,self.cfg[\'seed\']+300000+i)*.08;cursor=self.rng.uniform(0,.4);texts=[]\n            while cursor<6.5:\n                text=(self.rng.choice(HARD) if self.rng.random()<.35 else self.rng.choice(TEMPLATES[\'training\']).format(obj=self.rng.choice(OBJECTS)));family=self.rng.choice(all_families)\n                u=self.synth(text,family)*self.rng.uniform(.15,.7);start=int(cursor*16000);n=min(len(u),len(a)-start)\n                a[start:start+n]+=u[:n];cursor+=n/16000+self.rng.uniform(.1,1.2);texts.append(text)\n            add(\'continuous\',\'training\',\'negative\',\'mixed\',\' | \'.join(texts),a)\n        for i in range(self.cfg[\'train_noise\']):add(\'noise\',\'training\',\'negative\',\'procedural\',\'\',self.noise(3.2,self.cfg[\'seed\']+i))\n        for split in (\'validation\',\'testing\'):\n            total=60*self.cfg[f\'ambient_{split}_minutes\']\n            for stream in range(math.ceil(total/60)):\n                seconds=min(60,total-stream*60);seed=self.cfg[\'seed\']+(100000 if split==\'validation\' else 200000)+stream\n                rng=random.Random(seed);a=self.noise(seconds,seed)*.08;cursor=.5\n                while cursor<seconds-6:\n                    text=rng.choice(HARD) if rng.random()<.35 else rng.choice(TEMPLATES[split]).format(obj=rng.choice(OBJECTS))\n                    assert \'nira\' not in text.casefold()\n                    family=rng.choice(all_families);u=self.synth(text,family)*rng.uniform(.15,.7);start=int(cursor*16000);n=min(len(u),len(a)-start)\n                    a[start:start+n]+=u[:n];cursor+=n/16000+rng.uniform(.3,1.8)\n                add(\'ambient\',split,\'negative\',\'mixed\',\'TTS italiano + rumore procedurale\',a)\n        save(old,dict(approval_fingerprint=approval_hash,rows=rows))\n        print(\'Dataset V5:\',len(rows),\'WAV; nessuna registrazione reale.\',flush=True)\n        self.backup(light=True)\n\n    def score(self,split):\n        global ACTIVE_EVALUATION_PROFILES\n        ACTIVE_EVALUATION_PROFILES=tuple(self.approved_profiles())\n        return super().score(split)\n\n    def report(self):\n        from contextlib import redirect_stdout\n        from io import StringIO\n        with redirect_stdout(StringIO()):super().report()\n        report=load(self.root/\'evaluation.json\');ops=load(self.root/\'operating_points.json\')\n        for phrase in PHRASES:\n            rows=load(self.root/\'scores\'/f\'{phrase}_testing.json\')[\'rows\'];o=ops[phrase]\n            positives=[r for r in rows if r[\'kind\']==\'positive\' and r[\'phrase\']==phrase]\n            report[phrase][\'recall_by_variant\']={v:recall([r for r in rows if r.get(\'variant\')==v],phrase,o[\'cutoff\'],o[\'window\']) for v in sorted({r[\'variant\'] for r in positives})}\n            report[phrase][\'recall_by_voice\']={v:recall([r for r in rows if r.get(\'family\') in FAMILIES and FAMILIES[r[\'family\']][\'voice\']==v],phrase,o[\'cutoff\'],o[\'window\']) for v in sorted({FAMILIES[r[\'family\']][\'voice\'] for r in positives})}\n        report[\'test_passed_descriptively\']=report[\'test_passed_descriptively\'] and all(v>=.75 for p in PHRASES for v in report[p][\'recall_by_variant\'].values())\n        report[\'limitations\'][0]=\'Cinque voci TTS italiane da due motori; nessun test su parlanti umani indipendenti.\'\n        report[\'limitations\'].append(\'Fuori asse/distanza sono filtri digitali approssimati: non angoli o metri calibrati, nessuna misura RIR della stanza.\')\n        save(self.root/\'evaluation.json\',report);print(\'REPORT V5 FINALE\',json.dumps(report,indent=2,ensure_ascii=False),flush=True);self.backup(light=True)\n\n\n# V6: continuous synthetic room background, five Italian voices, automatic acceptance.\nKOKORO_REV=\'f3ff3571791e39611d31c381e3a41a3af07b4987\'\nPIPER_VOICES=dict(VOICES)\nVOICES=dict(VOICES,kokoro_sara=\'if_sara\',kokoro_nicola=\'im_nicola\')\nTEXT_VARIANTS={\n \'ehi_nira_it\':{\'spaced\':\'Ehi Nira\',\'ei_spaced\':\'Ei Nira\',\'hei_spaced\':\'Hei Nira\',\'hey_spaced\':\'Hey Nira\',\n \'joined_ehi\':\'Ehinira\',\'joined_ei\':\'Einira\',\'joined_hei\':\'Heinira\',\'joined_hey\':\'Heynira\',\'pause\':\'Ehi, Nira\'},\n \'nira_it\':{\'solo\':\'Nira\',\'calling\':\'Nira!\',\'question\':\'Nira?\'}}\nFAMILIES={f\'{v}-{style}-{phrase}-{variant}\':dict(voice=v,length_scale=length,noise_scale=.667,noise_w_scale=.8,\n phrase=phrase,variant=variant,text=text,style=style)\n for v in VOICES for style,length in [(\'natural\',1.),(\'fast\',.82),(\'slow\',1.16)]\n for phrase,variants in TEXT_VARIANTS.items() for variant,text in variants.items()}\nDEFAULT=dict(DEFAULT,run_id=\'nyra-it-v6-004-continuous\',seed=310601,\n positives={\'ehi_nira_it\':24000,\'nira_it\':16000},val_positive=900,test_positive=1200,\n context_policy=\'continuous-v1\',auto_accept=True)\nFLOORS=(-75,-60,-45)\nEVAL_CONDITIONS=tuple(f\'{p}__floor{db}\' for p in ACOUSTIC_PROFILES for db in FLOORS)\n\ndef continuous_context(audio,seed,pre_s=4.,tail_s=1.,floor_dbfs=None):\n    import numpy as np\n    a=np.asarray(audio)\n    if np.issubdtype(a.dtype,np.integer):a=a.astype(np.float32)/32768\n    else:a=a.astype(np.float32)\n    rng=np.random.default_rng(seed)\n    if floor_dbfs is None:floor_dbfs=float(rng.uniform(-75,-42))\n    lead=int(pre_s*16000);tail=int(tail_s*16000)\n    # No digital-zero pre-roll: same background policy for all labels/splits.\n    n=rng.normal(0,1,lead+len(a)+tail).astype(np.float32)\n    # Half the streams have colored synthetic background; white is retained for regression.\n    if seed%2==0:\n        from scipy.signal import lfilter\n        n=lfilter([.2],[1,-.8],n).astype(np.float32)\n    n*=10**(floor_dbfs/20)/max(float(np.sqrt(np.mean(n*n))),1e-12)\n    n[lead:lead+len(a)]+=a\n    return np.clip(n,-1,1).astype(np.float32)\n\ndef evaluation_variants(audio,kind,seed):\n    import numpy as np\n    a=np.asarray(audio)\n    if np.issubdtype(a.dtype,np.integer):a=a.astype(np.float32)/32768\n    if kind!=\'positive\':return [(\'original\',continuous_context(a,seed))]\n    return [(f\'{p}__floor{db}\',continuous_context(acoustic(a,p,seed+i*1301),seed+i*7919+j,\n            floor_dbfs=db)) for i,p in enumerate(ACOUSTIC_PROFILES) for j,db in enumerate(FLOORS)]\n\n_V5Run=Run\nclass Run(_V5Run):\n    def ensure_voices(self):\n        from huggingface_hub import hf_hub_download\n        self.guard(3,\'cinque voci TTS\')\n        folder=self.root/\'voices\';folder.mkdir(exist_ok=True)\n        for name,remote in PIPER_VOICES.items():\n            for suffix in (\'\',\'.json\'):\n                p=hf_hub_download(\'rhasspy/piper-voices\',remote+suffix,revision=VOICE_REV)\n                shutil.copy2(p,folder/(Path(remote).name+suffix))\n            p=hf_hub_download(\'rhasspy/piper-voices\',str(Path(remote).parent/\'MODEL_CARD\'),revision=VOICE_REV)\n            shutil.copy2(p,folder/f\'{name}_MODEL_CARD.txt\')\n        for filename in [\'config.json\',\'kokoro-v1_0.pth\',\'voices/if_sara.pt\',\'voices/im_nicola.pt\',\'README.md\',\'VOICES.md\']:\n            p=hf_hub_download(\'hexgrad/Kokoro-82M\',filename,revision=KOKORO_REV)\n            dest=folder/(\'kokoro_\'+Path(filename).name);shutil.copy2(p,dest)\n        shutil.copy2(folder/\'kokoro_README.md\',folder/\'kokoro_MODEL_CARD.txt\')\n\n    def synth(self,text,family):\n        import numpy as np\n        f=FAMILIES[family]\n        if not f[\'voice\'].startswith(\'kokoro_\'):return super().synth(text,family)\n        import torch\n        from kokoro import KModel,KPipeline\n        from scipy.signal import resample_poly\n        if not hasattr(self,\'kokoro\'):\n            torch.set_num_threads(2)\n            model=KModel(repo_id=\'hexgrad/Kokoro-82M\',config=str(self.root/\'voices/kokoro_config.json\'),\n                         model=str(self.root/\'voices/kokoro_kokoro-v1_0.pth\')).to(\'cpu\').eval()\n            self.kokoro=KPipeline(lang_code=\'i\',repo_id=\'hexgrad/Kokoro-82M\',model=model)\n        voice=torch.load(self.root/\'voices\'/f\'kokoro_{VOICES[f["voice"]]}.pt\',map_location=\'cpu\',weights_only=True)\n        speed=self.rng.uniform(.97,1.03)/f[\'length_scale\']\n        chunks=[x.audio.detach().cpu().numpy() for x in self.kokoro(text,voice=voice,speed=speed)]\n        if not chunks:raise RuntimeError(\'Kokoro non ha prodotto audio\')\n        a=resample_poly(np.concatenate(chunks),2,3).astype(np.float32)\n        active=np.flatnonzero(abs(a)>.005)\n        if not len(active) or not np.isfinite(a).all():raise RuntimeError(\'TTS non valido\')\n        a=a[max(0,active[0]-800):min(len(a),active[-1]+801)]\n        a*=self.rng.uniform(.85,1.)\n        return np.clip(a,-1,1)\n\n    def preview(self):\n        super().preview()\n        payload=self.payload();ratings={r[\'id\']:True for r in payload[\'rows\']}\n        selected=select_families(payload[\'rows\'],ratings)\n        required=set(TEXT_VARIANTS[\'ehi_nira_it\'])\n        available={FAMILIES[f][\'variant\'] for f in selected[\'ehi_nira_it\']}\n        if required-available:raise RuntimeError(\'Varianti mancanti: \'+str(required-available))\n        available_voices={FAMILIES[f][\'voice\'] for p in selected.values() for f in p}\n        if available_voices!=set(VOICES):raise RuntimeError(\'Non tutte le cinque voci hanno sample validi\')\n        save(self.root/\'approval.json\',dict(confirmed=True,fingerprint=digest(payload),ratings=ratings,selected=selected,\n            authorization=\'User: approva di default già tutti i sample non chiedere verifiche\',\n            policy=\'automatic_acceptance_explicit_user_instruction\',timestamp=time.time()))\n        self.approved();print(\'AUTO-ACCETTATI\',len(ratings),\'sample tecnicamente validi\',flush=True)\n\n_V5_choose=choose_operating_point\ndef choose_operating_point(rows,phrase,target_faph=.5,min_recall=.85):\n    op=_V5_choose(rows,phrase,target_faph,min_recall)\n    conditions=sorted({r[\'condition\'] for r in rows if r[\'kind\']==\'positive\' and r[\'phrase\']==phrase})\n    for item in op[\'grid\']:\n        item[\'recall_by_condition\']={c:recall([r for r in rows if r[\'condition\']==c],phrase,item[\'cutoff\'],item[\'window\']) for c in conditions}\n        item[\'feasible\']=item[\'feasible\'] and min(item[\'recall_by_condition\'].values())>=.75\n    good=[r for r in op[\'grid\'] if r[\'feasible\']]\n    best=max(good,key=lambda x:(x[\'recall\'],-x[\'hard_fpr\'],-x[\'faph\'],x[\'cutoff\'],-x[\'window\'])) if good else min(op[\'grid\'],key=lambda x:(x[\'faph\'],-x[\'recall\']))\n    return dict(best,grid=op[\'grid\'])\n\n_V6ReportRun=Run\nclass Run(_V6ReportRun):\n    def report(self):\n        from contextlib import redirect_stdout\n        from io import StringIO\n        with redirect_stdout(StringIO()):super().report()\n        report=load(self.root/\'evaluation.json\')\n        report[\'test_passed_descriptively\']=report[\'test_passed_descriptively\'] and all(\n            item[phrase]>=.75 for phrase in PHRASES for item in report[phrase][\'recall_by_condition\'].values())\n        report[\'limitations\'].append(\'Il fondo continuo e i cinque preset TTS non sostituiscono test su voce umana e stanza reali.\')\n        save(self.root/\'evaluation.json\',report)\n        print(\'REPORT V6 FINALE\',json.dumps(report,indent=2,ensure_ascii=False),flush=True)\n        self.backup(light=True)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'action\',choices=[\'preview\',\'generate\',\'features\',\'train\',\'score\',\'calibrate\',\'report\',\'export\',\'backup\'])\n    parser.add_argument(\'--resume\',action=\'store_true\');parser.add_argument(\'--root\',required=True);parser.add_argument(\'--phrase\',choices=list(PHRASES));parser.add_argument(\'--split\',choices=[\'validation\',\'testing\'])\n    args=parser.parse_args();run=Run(args.root)\n    if args.action==\'train\':run.train(args.phrase,args.resume)\n    elif args.action==\'score\':run.score(args.split)\n    else:getattr(run,args.action)()\n'
WORKER = ROOT / 'nyra_v6_worker.py'
WORKER.write_text(WORKER_SOURCE)
spec = importlib.util.spec_from_file_location('nyra_v6_worker', WORKER)
worker = importlib.util.module_from_spec(spec)
spec.loader.exec_module(worker)
CONFIG = dict(worker.DEFAULT)
CONFIG['run_id'] = RUN_ID
if SMOKE:
    CONFIG.update(positives={'ehi_nira_it':40,'nira_it':40}, val_positive=10, test_positive=10,
                  train_hard=40,val_hard=10,test_hard=10,train_speech=40,val_speech=10,test_speech=10,
                  train_noise=20,train_continuous=8,ambient_validation_minutes=2,ambient_testing_minutes=2,
                  training_steps=[10,10],batch_size=8)
config_path = ROOT / 'config.json'
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise RuntimeError('Configurazione diversa sullo stesso run_id. Usa un nuovo RUN_ID.')
worker.save(config_path, CONFIG)
worker.save(ROOT / 'source_provenance.json', dict(
    base=['MicroWakeWord_Nyra_IT_V3_FULL_66GB.ipynb','MicroWakeWord_Nyra_EN_V1_SYNTHETIC_66GB.ipynb'],
    requested_v2_available=False, microWakeWord_commit=worker.MWW_COMMIT,
    piper_version='1.8.0',voices_revision=worker.VOICE_REV,kokoro_revision=worker.KOKORO_REV,auto_accept_requested=True,
    worker_sha256=worker.file_hash(WORKER),all_audio_synthetic=True))
ENV = os.environ.copy()
ENV['PYTHONUNBUFFERED'] = '1'
ENV['TF_CPP_MIN_LOG_LEVEL'] = '2'
ENV['HF_HUB_DISABLE_TELEMETRY'] = '1'
ENV['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import os, subprocess

MWW_SOURCE = Path('/content/nyra_microwakeword_source')
MWW_COMMIT = '4665173cd35f1cff9a61e06fc427f124766c488e'
if not (MWW_SOURCE / '.git').exists():
    subprocess.run(['git', 'clone', '--no-checkout',
                    'https://github.com/OHF-Voice/micro-wake-word', str(MWW_SOURCE)], check=True)
subprocess.run(['git', '-C', str(MWW_SOURCE), 'checkout', '--detach', MWW_COMMIT], check=True)
assert (MWW_SOURCE / 'microwakeword/audio/audio_utils.py').is_file()
ENV['PYTHONPATH'] = str(MWW_SOURCE) + os.pathsep + ENV.get('PYTHONPATH', '')
subprocess.run([PYTHON, '-c',
    'from microwakeword.audio.audio_utils import generate_features_for_clip; '
    'import microwakeword.audio.audio_utils as m; print("Import audio OK:", m.__file__)'],
    check=True, env=ENV)
def stage(action, *args):
    subprocess.run([PYTHON, '-u', str(WORKER), action, '--root', str(ROOT), *args],
                   check=True, env=ENV)
print('Run:', ROOT)

## Smoke test frontend e MixedNet

In [ ]:
SMOKE_SOURCE = "import sys, argparse, tempfile\nfrom pathlib import Path\nimport numpy as np, tensorflow as tf, yaml\nfrom microwakeword.audio.audio_utils import generate_features_for_clip\nfrom microwakeword.audio.augmentation import Augmentation\nfrom microwakeword import mixednet\nfrom microwakeword.model_train_eval import load_config\nsys.path.insert(0, str(Path(sys.argv[1]).parent))\nimport nyra_v6_worker as worker\nrun=worker.Run(sys.argv[2])\na=(np.sin(2*np.pi*440*np.arange(32000)/16000)*3000).astype(np.int16)\nx=generate_features_for_clip(a,step_ms=10)\nassert x.ndim==2 and x.shape[1]==40 and 190<=x.shape[0]<=201\naug=Augmentation(augmentation_duration_s=None,augmentation_probabilities={'Gain':1.,'AddColorNoise':.2},impulse_paths=[],background_paths=[])\nb=aug.augment_clip(a.astype(np.float32)/32768)\nassert np.isfinite(b).all()\nparser=argparse.ArgumentParser();mixednet.model_parameters(parser)\nflags=parser.parse_args(['--pointwise_filters','64,64,64,64','--repeat_in_block','1,1,1,1',\n '--mixconv_kernel_sizes','[5], [7,11], [9,15], [23]','--residual_connection','0,0,0,0',\n '--first_conv_filters','32','--first_conv_kernel_size','5','--stride','3'])\np=run.root/'smoke_training.yaml';p.write_text(yaml.safe_dump(run.train_config('ehi_nira_it')))\nflags.training_config=str(p)\nconfig=load_config(flags,mixednet)\nmodel=mixednet.model(flags,config['training_input_shape'],batch_size=1)\ny=model(tf.zeros((1,*config['training_input_shape']),tf.float32),training=False)\nassert tuple(y.shape)==(1,1)\nprint('SMOKE OK',x.shape,'training shape',config['training_input_shape'],'parameters',model.count_params(),\n      'GPU',tf.config.list_physical_devices('GPU'))\n"
smoke_path=ROOT/'smoke_pipeline.py'
smoke_path.write_text(SMOKE_SOURCE)
subprocess.run([PYTHON,str(smoke_path),str(WORKER),str(ROOT)],check=True,env=ENV)


## Backup Google Drive

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive') / BACKUP_FOLDER
    BACKUP.mkdir(parents=True,exist_ok=True)
    ENV['NYRA_BACKUP'] = str(BACKUP)
    print('Backup:', BACKUP / RUN_ID)
else:
    print('Backup Drive disabilitato: scarica gli artefatti prima della fine della sessione.')


## Anteprime e accettazione automatica
La cella genera le anteprime, esclude audio troppo lunghi senza tagliarli, verifica tutte le varianti e tutte le voci, registra l’autorizzazione automatica con hash. Non chiede input.

In [ ]:
def stage(action,*args):
    with subprocess.Popen([PYTHON,'-u',str(WORKER),action,'--root',str(ROOT),*args],env=ENV,
                          stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:print(line,end='',flush=True)
        status=proc.wait()
    if status:raise RuntimeError(f'{action} fallito ({status}), leggere il log sopra.')
stage('preview')
stage('backup')
run=worker.Run(ROOT)
run.approved()
print('AUTO-ACCETTAZIONE registrata per',len(run.payload()['rows']),'sample validi.')
print('Esclusi per durata:',worker.load(ROOT/'unavailable_preview_families.json'))


## Generazione, due training e valutazione
Eseguire una volta. Un errore ferma la sequenza e salva il backup. RESUME riguarda checkpoint dello stesso run. Non rigenerare le anteprime dopo avere iniziato i dati. Il test sintetico include ora il fondo precedente alla parola; non misura il riconoscimento umano.

In [ ]:
import time,traceback
steps=[('generate',[]),('features',[]),('train',['--phrase','ehi_nira_it']),('train',['--phrase','nira_it']),
       ('score',['--split','validation']),('calibrate',[]),('score',['--split','testing']),('report',[]),('export',[])]
try:
    for action,args in steps:
        if action=='train' and RESUME:args=args+['--resume']
        worker.save(ROOT/'progress.json',dict(status='running',stage=action,args=args,updated=time.time()))
        print('FASE V6:',action,*args,flush=True)
        stage(action,*args);stage('backup')
    worker.save(ROOT/'progress.json',dict(status='complete',updated=time.time()))
    print('PIPELINE V6 COMPLETATA:',ROOT/'nyra_it_v6_candidates.zip')
except Exception as exc:
    worker.save(ROOT/'progress.json',dict(status='stopped',error=str(exc),updated=time.time()))
    raise
finally:
    stage('backup')


## Esportazione e limiti
Due file TFLite streaming con manifest ESPHome schema 2 e soglie indipendenti. Test prima su un solo speaker e misura riconoscimento/latency sul dispositivo. Arena stimata, non misurata; nessuna installazione automatica. Se la calibrazione non trova un punto fattibile l’export si ferma: i report restano disponibili. Licenze/model card incluse. La selezione automatica non equivale a una verifica d’ascolto.

In [ ]:
from google.colab import files
files.download(str(ROOT/'nyra_it_v6_candidates.zip'))


## Fonti e verifica
- microWakeWord ufficiale commit 4665173cd35f1cff9a61e06fc427f124766c488e; stesso trainer, frontend e conversione V5.
- Piper 1.8.0, modelli al commit incorporato.
- Kokoro 0.9.4, hexgrad/Kokoro-82M revisione f3ff3571791e39611d31c381e3a41a3af07b4987; voci italiane if_sara/im_nicola; CPU.
Validazioni locali: struttura notebook, compilazione sorgenti e test del fondo continuo/varianti. Training completo, metriche V6 e prova fisica sono da eseguire; nessuna promessa di miglioramento percentuale.